In [32]:
import pandas as pd

# Replace these with the actual paths to your CSV files
lt_sf= '/content/DOST complete rows - LT SF.csv'
lt_balili = '/content/DOST complete rows - LT BALILI.csv'
atok = '/content/DOST complete rows - ATOK.csv'
com_field_data = '/content/combined_field_data.csv'

# Load each CSV into a DataFrame
lt_sf_df1 = pd.read_csv(lt_sf)
lt_balili_df2 = pd.read_csv(lt_balili)
atok_df3 = pd.read_csv(atok)
com_field_data_df4 = pd.read_csv(com_field_data)

print("First 5 rows of df1:")
display(lt_sf_df1.head())
print(lt_sf_df1 .shape)

print("\nFirst 5 rows of df2:")
display(lt_balili_df2.head())

print("\nFirst 5 rows of df3:")
display(atok_df3.head())

print("\nFirst 5 rows of df4:")
display(com_field_data_df4.head())

First 5 rows of df1:


,Count,ph,k,p,n
0,1,6.4,1,0,0
1,2,6.4,1,0,0
2,3,6.0,0,0,0
3,4,6.8,0,0,0
4,5,6.8,0,0,0


(30, 5)

First 5 rows of df2:


,count,ph,k,p,n
0,1,6.4,1,0,0
1,2,6.4,0,0,0
2,3,6.4,0,0,0
3,4,6.4,0,0,0
4,5,6.8,0,0,0



First 5 rows of df3:


,Count,ph,k,p,n
0,1,6.4,2,0.0,0.0
1,1,6.4,NaN,0.0,NaN
2,2,6.4,2,0.0,1.0
3,3,6.4,2,1.0,0.0
4,4,6.4,2,1.0,1.0



First 5 rows of df4:


,uuid,spot_number,shot_number,shots_in_spot,image_filename,image_width,image_height,image_quality,capture_datetime,latitude,...,camera_heading,municipality,barangay,farm_name,crops,temperature_c,humidity_percent,notes,device_id,capture_mode
0,aeeed204,1,1,10,images/2026/01/21/latrinidad_balili_bsuback_20...,1080.0,1080.0,1080p,2026-01-21T00:31:27.435Z,16.453933,...,4.0,La Trinidad,Balili,BSU Back,"cabbage,custom_chinese_cabbage",16.9,79.0,NaN,TECNO_TECNO_CM8_5EAA20,field
1,a18676dc,1,2,10,images/2026/01/21/latrinidad_balili_bsuback_20...,1080.0,1080.0,1080p,2026-01-21T00:31:30.356Z,16.453933,...,355.0,La Trinidad,Balili,BSU Back,"cabbage,custom_chinese_cabbage",16.9,79.0,NaN,TECNO_TECNO_CM8_5EAA20,field
2,f781136b,1,3,10,images/2026/01/21/latrinidad_balili_bsuback_20...,1080.0,1080.0,1080p,2026-01-21T00:31:33.202Z,16.453933,...,346.0,La Trinidad,Balili,BSU Back,"cabbage,custom_chinese_cabbage",16.9,79.0,NaN,TECNO_TECNO_CM8_5EAA20,field
3,7cfb041f,1,4,10,images/2026/01/21/latrinidad_balili_bsuback_20...,1080.0,1080.0,1080p,2026-01-21T00:31:37.907Z,16.453933,...,331.0,La Trinidad,Balili,BSU Back,"cabbage,custom_chinese_cabbage",16.9,79.0,NaN,TECNO_TECNO_CM8_5EAA20,field
4,db27f1eb,1,5,10,images/2026/01/21/latrinidad_balili_bsuback_20...,1080.0,1080.0,1080p,2026-01-21T00:31:40.736Z,16.453933,...,342.0,La Trinidad,Balili,BSU Back,"cabbage,custom_chinese_cabbage",16.9,79.0,NaN,TECNO_TECNO_CM8_5EAA20,field


In [33]:
print("Columns of lt_sf_df1:")
print(lt_sf_df1.columns.tolist())

print("\nColumns of lt_balili_df2:")
print(lt_balili_df2.columns.tolist())

print("\nColumns of atok_df3:")
print(atok_df3.columns.tolist())

print("\nColumns of com_field_data_df4:")
print(com_field_data_df4.columns.tolist())

Columns of lt_sf_df1:
['Count', 'ph', 'k', 'p', 'n']

Columns of lt_balili_df2:
['count', 'ph', 'k', 'p', 'n']

Columns of atok_df3:
['Count', 'ph', 'k', 'p', 'n']

Columns of com_field_data_df4:
['uuid', 'spot_number', 'shot_number', 'shots_in_spot', 'image_filename', 'image_width', 'image_height', 'image_quality', 'capture_datetime', 'latitude', 'longitude', 'altitude_m', 'altitude_accuracy_m', 'gps_accuracy_m', 'gps_reading_count', 'camera_pitch', 'camera_roll', 'camera_heading', 'municipality', 'barangay', 'farm_name', 'crops', 'temperature_c', 'humidity_percent', 'notes', 'device_id', 'capture_mode']


#Data Merging

In [56]:
import numpy as np
import pandas as pd

# 1. Initialize the final DataFrame with all rows from com_field_data_df4
final_merged_df = com_field_data_df4.copy()

# --- 2. Prepare lt_*_df dataframes to be unique on 'spot_number' by aggregating duplicates ---
# If 'Count' has duplicates in the source dataframes, we'll take the mean of 'ph', 'k', 'p', 'n'
# This prevents row duplication when merging.

# Helper function to convert columns to numeric and then group, and explicitly rename
def process_lt_df(df, count_col_name, suffix=""):
    # Convert relevant columns to numeric, coercing errors to NaN
    for col in ['ph', 'k', 'p', 'n']:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')
    # Group by count_col_name and take mean of numeric columns
    unique_df = df.groupby(count_col_name)[['ph', 'k', 'p', 'n']].mean().reset_index()
    unique_df = unique_df.rename(columns={count_col_name: 'spot_number'})
    # Explicitly rename the ph, k, p, n columns for unambiguous merging
    if suffix:
        unique_df = unique_df.rename(columns={col: f"{col}{suffix}" for col in ['ph', 'k', 'p', 'n']})
    return unique_df

# Process lt_sf_df1 and rename its columns before merging
lt_sf_unique = process_lt_df(lt_sf_df1.copy(), 'Count', '_sf_merged')

# Process lt_balili_df2 and rename its columns before merging
lt_balili_df2_temp = lt_balili_df2.rename(columns={'count': 'Count'}) # Rename 'count' to 'Count' first
lt_balili_unique = process_lt_df(lt_balili_df2_temp.copy(), 'Count', '_balili_merged')

# Process atok_df3 and rename its columns before merging
atok_unique = process_lt_df(atok_df3.copy(), 'Count', '_atok_merged')

# --- 3. Perform multiple left merges to bring in data ---
# No suffixes needed in merge calls as columns are already uniquely named
final_merged_df = pd.merge(final_merged_df, lt_sf_unique, on='spot_number', how='left')
final_merged_df = pd.merge(final_merged_df, lt_balili_unique, on='spot_number', how='left')
final_merged_df = pd.merge(final_merged_df, atok_unique, on='spot_number', how='left')

# --- 4. Conditionally populate final 'ph', 'k', 'p', 'n' columns ---
# Initialize final ph, k, p, n columns as NaN
final_merged_df['ph'] = np.nan
final_merged_df['k'] = np.nan
final_merged_df['p'] = np.nan
final_merged_df['n'] = np.nan

# Define conditions and corresponding choices for each 'ph', 'k', 'p', 'n' column.
# The order of conditions in np.select matters for precedence (SF > Balili > Atok if overlapping).
conditions = [
    (final_merged_df['farm_name'] == 'SF') | (final_merged_df['farm_name'] == 'Strawberry Farm'),
    final_merged_df['barangay'] == 'Balili',
    final_merged_df['municipality'] == 'Atok'
]

# Choices for 'ph' column
choices_ph = [
    final_merged_df['ph_sf_merged'],
    final_merged_df['ph_balili_merged'],
    final_merged_df['ph_atok_merged']
]
final_merged_df['ph'] = np.select(conditions, choices_ph, default=np.nan)

# Choices for 'k' column
choices_k = [
    final_merged_df['k_sf_merged'],
    final_merged_df['k_balili_merged'],
    final_merged_df['k_atok_merged']
]
final_merged_df['k'] = np.select(conditions, choices_k, default=np.nan)

# Choices for 'p' column
choices_p = [
    final_merged_df['p_sf_merged'],
    final_merged_df['p_balili_merged'],
    final_merged_df['p_atok_merged']
]
final_merged_df['p'] = np.select(conditions, choices_p, default=np.nan)

# Choices for 'n' column
choices_n = [
    final_merged_df['n_sf_merged'],
    final_merged_df['n_balili_merged'],
    final_merged_df['n_atok_merged']
]
final_merged_df['n'] = np.select(conditions, choices_n, default=np.nan)

# --- 5. Clean up temporary columns ---
# Identify and drop all temporary merged columns (e.g., ph_sf_merged)
temp_merge_cols = [col for col in final_merged_df.columns if '_merged' in col]
final_merged_df = final_merged_df.drop(columns=temp_merge_cols)

print("Final DataFrame with all rows from com_field_data_df4 and conditionally appended 'ph', 'k', 'p', 'n' columns (no row duplication):")
display(final_merged_df.head())
print(f"Number of rows in this new final DataFrame: {len(final_merged_df)}")
print(f"Shape of original com_field_data_df4: {com_field_data_df4.shape}")
print(f"Shape of final_merged_df: {final_merged_df.shape}")

# Save the combined DataFrame to a CSV file
output_csv_path_left_merge_no_dup = '/content/final_combined_data_merged.csv'
final_merged_df.to_csv(output_csv_path_left_merge_no_dup, index=False)
print(f"\nCombined data (left merge, no duplication) saved to {output_csv_path_left_merge_no_dup}")

Final DataFrame with all rows from com_field_data_df4 and conditionally appended 'ph', 'k', 'p', 'n' columns (no row duplication):


,uuid,spot_number,shot_number,shots_in_spot,image_filename,image_width,image_height,image_quality,capture_datetime,latitude,...,temperature_c,humidity_percent,notes,device_id,capture_mode,location_sort_key,ph,k,p,n
0,aeeed204,1,1,10,images/2026/01/21/latrinidad_balili_bsuback_20...,1080.0,1080.0,1080p,2026-01-21T00:31:27.435Z,16.453933,...,16.9,79.0,NaN,TECNO_TECNO_CM8_5EAA20,field,0,6.4,1.0,0.0,0.0
1,a18676dc,1,2,10,images/2026/01/21/latrinidad_balili_bsuback_20...,1080.0,1080.0,1080p,2026-01-21T00:31:30.356Z,16.453933,...,16.9,79.0,NaN,TECNO_TECNO_CM8_5EAA20,field,0,6.4,1.0,0.0,0.0
2,f781136b,1,3,10,images/2026/01/21/latrinidad_balili_bsuback_20...,1080.0,1080.0,1080p,2026-01-21T00:31:33.202Z,16.453933,...,16.9,79.0,NaN,TECNO_TECNO_CM8_5EAA20,field,0,6.4,1.0,0.0,0.0
3,7cfb041f,1,4,10,images/2026/01/21/latrinidad_balili_bsuback_20...,1080.0,1080.0,1080p,2026-01-21T00:31:37.907Z,16.453933,...,16.9,79.0,NaN,TECNO_TECNO_CM8_5EAA20,field,0,6.4,1.0,0.0,0.0
4,db27f1eb,1,5,10,images/2026/01/21/latrinidad_balili_bsuback_20...,1080.0,1080.0,1080p,2026-01-21T00:31:40.736Z,16.453933,...,16.9,79.0,NaN,TECNO_TECNO_CM8_5EAA20,field,0,6.4,1.0,0.0,0.0


Number of rows in this new final DataFrame: 1787
Shape of original com_field_data_df4: (1787, 28)
Shape of final_merged_df: (1787, 32)

Combined data (left merge, no duplication) saved to /content/final_combined_data_merged.csv


In [42]:
import numpy as np

# Define the custom order for sorting locations
# This creates a combined location identifier for sorting precedence
def get_location_sort_key(row):
    if row['barangay'] == 'Balili':
        return 0 # Balili comes first
    elif row['farm_name'] == 'SF' or row['farm_name'] == 'Strawberry Farm':
        return 1 # SF/Strawberry Farm comes second
    elif row['municipality'] == 'Atok':
        return 2 # Atok comes third
    else:
        return 3 # Other locations last

# Apply the function to create a temporary sorting column
final_merged_df['location_sort_key'] = final_merged_df.apply(get_location_sort_key, axis=1)

# Sort the DataFrame
sorted_final_merged_df = final_merged_df.sort_values(by=['location_sort_key', 'spot_number'], ascending=[True, True])

# Drop the temporary sorting column
sorted_final_merged_df = sorted_final_merged_df.drop(columns=['location_sort_key'])

print("Final DataFrame sorted by location precedence and spot_number:")
display(sorted_final_merged_df.head())
print(f"Shape of sorted_final_merged_df: {sorted_final_merged_df.shape}")

Final DataFrame sorted by location precedence and spot_number:


,uuid,spot_number,shot_number,shots_in_spot,image_filename,image_width,image_height,image_quality,capture_datetime,latitude,...,crops,temperature_c,humidity_percent,notes,device_id,capture_mode,ph,k,p,n
0,aeeed204,1,1,10,images/2026/01/21/latrinidad_balili_bsuback_20...,1080.0,1080.0,1080p,2026-01-21T00:31:27.435Z,16.453933,...,"cabbage,custom_chinese_cabbage",16.9,79.0,NaN,TECNO_TECNO_CM8_5EAA20,field,6.4,1.0,0.0,0.0
1,a18676dc,1,2,10,images/2026/01/21/latrinidad_balili_bsuback_20...,1080.0,1080.0,1080p,2026-01-21T00:31:30.356Z,16.453933,...,"cabbage,custom_chinese_cabbage",16.9,79.0,NaN,TECNO_TECNO_CM8_5EAA20,field,6.4,1.0,0.0,0.0
2,f781136b,1,3,10,images/2026/01/21/latrinidad_balili_bsuback_20...,1080.0,1080.0,1080p,2026-01-21T00:31:33.202Z,16.453933,...,"cabbage,custom_chinese_cabbage",16.9,79.0,NaN,TECNO_TECNO_CM8_5EAA20,field,6.4,1.0,0.0,0.0
3,7cfb041f,1,4,10,images/2026/01/21/latrinidad_balili_bsuback_20...,1080.0,1080.0,1080p,2026-01-21T00:31:37.907Z,16.453933,...,"cabbage,custom_chinese_cabbage",16.9,79.0,NaN,TECNO_TECNO_CM8_5EAA20,field,6.4,1.0,0.0,0.0
4,db27f1eb,1,5,10,images/2026/01/21/latrinidad_balili_bsuback_20...,1080.0,1080.0,1080p,2026-01-21T00:31:40.736Z,16.453933,...,"cabbage,custom_chinese_cabbage",16.9,79.0,NaN,TECNO_TECNO_CM8_5EAA20,field,6.4,1.0,0.0,0.0


Shape of sorted_final_merged_df: (1787, 31)


In [43]:
# Apply the function to create a temporary sorting column on com_field_data_df4
com_field_data_df4['location_sort_key'] = com_field_data_df4.apply(get_location_sort_key, axis=1)

# Sort com_field_data_df4
sorted_com_field_data_df4 = com_field_data_df4.sort_values(by=['location_sort_key', 'spot_number'], ascending=[True, True])

# Drop the temporary sorting column from com_field_data_df4 as well, to keep it clean
sorted_com_field_data_df4 = sorted_com_field_data_df4.drop(columns=['location_sort_key'])

print("Original com_field_data_df4 sorted by location precedence and spot_number:")
display(sorted_com_field_data_df4.head())
print(f"Shape of sorted_com_field_data_df4: {sorted_com_field_data_df4.shape}")

Original com_field_data_df4 sorted by location precedence and spot_number:


,uuid,spot_number,shot_number,shots_in_spot,image_filename,image_width,image_height,image_quality,capture_datetime,latitude,...,camera_heading,municipality,barangay,farm_name,crops,temperature_c,humidity_percent,notes,device_id,capture_mode
0,aeeed204,1,1,10,images/2026/01/21/latrinidad_balili_bsuback_20...,1080.0,1080.0,1080p,2026-01-21T00:31:27.435Z,16.453933,...,4.0,La Trinidad,Balili,BSU Back,"cabbage,custom_chinese_cabbage",16.9,79.0,NaN,TECNO_TECNO_CM8_5EAA20,field
1,a18676dc,1,2,10,images/2026/01/21/latrinidad_balili_bsuback_20...,1080.0,1080.0,1080p,2026-01-21T00:31:30.356Z,16.453933,...,355.0,La Trinidad,Balili,BSU Back,"cabbage,custom_chinese_cabbage",16.9,79.0,NaN,TECNO_TECNO_CM8_5EAA20,field
2,f781136b,1,3,10,images/2026/01/21/latrinidad_balili_bsuback_20...,1080.0,1080.0,1080p,2026-01-21T00:31:33.202Z,16.453933,...,346.0,La Trinidad,Balili,BSU Back,"cabbage,custom_chinese_cabbage",16.9,79.0,NaN,TECNO_TECNO_CM8_5EAA20,field
3,7cfb041f,1,4,10,images/2026/01/21/latrinidad_balili_bsuback_20...,1080.0,1080.0,1080p,2026-01-21T00:31:37.907Z,16.453933,...,331.0,La Trinidad,Balili,BSU Back,"cabbage,custom_chinese_cabbage",16.9,79.0,NaN,TECNO_TECNO_CM8_5EAA20,field
4,db27f1eb,1,5,10,images/2026/01/21/latrinidad_balili_bsuback_20...,1080.0,1080.0,1080p,2026-01-21T00:31:40.736Z,16.453933,...,342.0,La Trinidad,Balili,BSU Back,"cabbage,custom_chinese_cabbage",16.9,79.0,NaN,TECNO_TECNO_CM8_5EAA20,field


Shape of sorted_com_field_data_df4: (1787, 27)


In [45]:
# Get the list of original columns from com_field_data_df4 to compare
common_cols_to_check = sorted_com_field_data_df4.columns.tolist()

# Prepare sorted_com_field_data_df4 for comparison: sort by uuid and reset index
df4_for_comparison = sorted_com_field_data_df4.sort_values(by='uuid').reset_index(drop=True)

# Prepare sorted_final_merged_df for comparison: select common columns, sort by uuid, and reset index
merged_for_comparison = sorted_final_merged_df[common_cols_to_check].sort_values(by='uuid').reset_index(drop=True)

# Perform the integrity check using .equals()
integrity_check_passed = df4_for_comparison.equals(merged_for_comparison)

if integrity_check_passed:
    print("Data integrity check passed: The original columns of `com_field_data_df4` are unchanged in `sorted_final_merged_df` for matching UUIDs.")
else:
    print("Data integrity check failed: Discrepancies found in the original columns.")
    # Find and display the differences if any
    differences = df4_for_comparison.compare(merged_for_comparison)
    if not differences.empty:
        print("Rows and columns with discrepancies:")
        display(differences)
    else:
        print("No apparent value differences in original columns, but pandas .equals() returned False. This might be due to differences in column dtypes or metadata, which .equals() checks strictly. However, the values themselves should be identical.")

Data integrity check passed: The original columns of `com_field_data_df4` are unchanged in `sorted_final_merged_df` for matching UUIDs.


In [46]:
# Save the sorted original com_field_data_df4
output_path_df4_sorted = '/content/com_field_data_df4_sorted.csv'
sorted_com_field_data_df4.to_csv(output_path_df4_sorted, index=False)
print(f"Sorted original com_field_data_df4 saved to: {output_path_df4_sorted}")

Sorted original com_field_data_df4 saved to: /content/com_field_data_df4_sorted.csv


In [47]:
# Save the merged and sorted DataFrame
output_path_merged_sorted = '/content/final_merged_data_sorted.csv'
sorted_final_merged_df.to_csv(output_path_merged_sorted, index=False)
print(f"Merged and sorted final DataFrame saved to: {output_path_merged_sorted}")

Merged and sorted final DataFrame saved to: /content/final_merged_data_sorted.csv


#Data cleaning

removing null rows
- n
- p
- k
- ph

In [53]:
import pandas as pd

# Load the merged and sorted DataFrame
input_path = '/content/final_merged_data_sorted.csv'
df_to_clean = pd.read_csv(input_path)

print("Original DataFrame shape:", df_to_clean.shape)
print("Original DataFrame head:")
display(df_to_clean.head())

# Define the columns to check for null values
columns_to_check = ['n', 'p', 'k', 'ph']

# Drop rows where any of the specified columns have NaN values
df_cleaned = df_to_clean.dropna(subset=columns_to_check)

print("\nCleaned DataFrame shape:", df_cleaned.shape)
print("Cleaned DataFrame head:")
display(df_cleaned.head())

# Save the cleaned DataFrame to a new CSV file
output_path_cleaned = '/content/final_merged_data_cleaned.csv'
df_cleaned.to_csv(output_path_cleaned, index=False)
print(f"\nCleaned data saved to: {output_path_cleaned}")

Original DataFrame shape: (1787, 31)
Original DataFrame head:


,uuid,spot_number,shot_number,shots_in_spot,image_filename,image_width,image_height,image_quality,capture_datetime,latitude,...,crops,temperature_c,humidity_percent,notes,device_id,capture_mode,ph,k,p,n
0,aeeed204,1,1,10,images/2026/01/21/latrinidad_balili_bsuback_20...,1080.0,1080.0,1080p,2026-01-21T00:31:27.435Z,16.453933,...,"cabbage,custom_chinese_cabbage",16.9,79.0,NaN,TECNO_TECNO_CM8_5EAA20,field,6.4,1.0,0.0,0.0
1,a18676dc,1,2,10,images/2026/01/21/latrinidad_balili_bsuback_20...,1080.0,1080.0,1080p,2026-01-21T00:31:30.356Z,16.453933,...,"cabbage,custom_chinese_cabbage",16.9,79.0,NaN,TECNO_TECNO_CM8_5EAA20,field,6.4,1.0,0.0,0.0
2,f781136b,1,3,10,images/2026/01/21/latrinidad_balili_bsuback_20...,1080.0,1080.0,1080p,2026-01-21T00:31:33.202Z,16.453933,...,"cabbage,custom_chinese_cabbage",16.9,79.0,NaN,TECNO_TECNO_CM8_5EAA20,field,6.4,1.0,0.0,0.0
3,7cfb041f,1,4,10,images/2026/01/21/latrinidad_balili_bsuback_20...,1080.0,1080.0,1080p,2026-01-21T00:31:37.907Z,16.453933,...,"cabbage,custom_chinese_cabbage",16.9,79.0,NaN,TECNO_TECNO_CM8_5EAA20,field,6.4,1.0,0.0,0.0
4,db27f1eb,1,5,10,images/2026/01/21/latrinidad_balili_bsuback_20...,1080.0,1080.0,1080p,2026-01-21T00:31:40.736Z,16.453933,...,"cabbage,custom_chinese_cabbage",16.9,79.0,NaN,TECNO_TECNO_CM8_5EAA20,field,6.4,1.0,0.0,0.0



Cleaned DataFrame shape: (1636, 31)
Cleaned DataFrame head:


,uuid,spot_number,shot_number,shots_in_spot,image_filename,image_width,image_height,image_quality,capture_datetime,latitude,...,crops,temperature_c,humidity_percent,notes,device_id,capture_mode,ph,k,p,n
0,aeeed204,1,1,10,images/2026/01/21/latrinidad_balili_bsuback_20...,1080.0,1080.0,1080p,2026-01-21T00:31:27.435Z,16.453933,...,"cabbage,custom_chinese_cabbage",16.9,79.0,NaN,TECNO_TECNO_CM8_5EAA20,field,6.4,1.0,0.0,0.0
1,a18676dc,1,2,10,images/2026/01/21/latrinidad_balili_bsuback_20...,1080.0,1080.0,1080p,2026-01-21T00:31:30.356Z,16.453933,...,"cabbage,custom_chinese_cabbage",16.9,79.0,NaN,TECNO_TECNO_CM8_5EAA20,field,6.4,1.0,0.0,0.0
2,f781136b,1,3,10,images/2026/01/21/latrinidad_balili_bsuback_20...,1080.0,1080.0,1080p,2026-01-21T00:31:33.202Z,16.453933,...,"cabbage,custom_chinese_cabbage",16.9,79.0,NaN,TECNO_TECNO_CM8_5EAA20,field,6.4,1.0,0.0,0.0
3,7cfb041f,1,4,10,images/2026/01/21/latrinidad_balili_bsuback_20...,1080.0,1080.0,1080p,2026-01-21T00:31:37.907Z,16.453933,...,"cabbage,custom_chinese_cabbage",16.9,79.0,NaN,TECNO_TECNO_CM8_5EAA20,field,6.4,1.0,0.0,0.0
4,db27f1eb,1,5,10,images/2026/01/21/latrinidad_balili_bsuback_20...,1080.0,1080.0,1080p,2026-01-21T00:31:40.736Z,16.453933,...,"cabbage,custom_chinese_cabbage",16.9,79.0,NaN,TECNO_TECNO_CM8_5EAA20,field,6.4,1.0,0.0,0.0



Cleaned data saved to: /content/final_merged_data_cleaned.csv


In [54]:
print("Summary Statistics for df_cleaned (Numerical Columns):")
display(df_cleaned.describe())

print("\nDataFrame Information for df_cleaned:")
df_cleaned.info()


Summary Statistics for df_cleaned (Numerical Columns):


,spot_number,shot_number,shots_in_spot,image_width,image_height,latitude,longitude,altitude_m,altitude_accuracy_m,gps_accuracy_m,gps_reading_count,camera_pitch,camera_roll,camera_heading,temperature_c,humidity_percent,ph,k,p,n
count,1636.000000,1636.000000,1636.000000,1428.000000,1428.000000,1189.000000,1189.000000,1189.000000,1189.000000,1189.000000,1189.0,1428.000000,1428.000000,1428.000000,1181.000000,1181.000000,1636.000000,1636.000000,1636.000000,1636.000000
mean,49.936430,3.864303,6.803178,1481.372549,1071.176471,16.514815,120.650282,1734.835156,10.616484,11.840875,1.0,21.471289,-2.737395,209.810224,18.791279,53.443692,6.630929,0.694988,0.938875,0.080685
std,48.370369,2.383926,2.338957,415.977088,55.684714,0.083510,0.080146,507.829324,7.394324,31.822884,0.0,16.998450,16.639138,101.009958,2.911753,15.786862,0.418761,0.707668,0.886145,0.272433
min,1.000000,1.000000,2.000000,1080.000000,720.000000,16.452461,120.590377,1358.000000,1.000000,1.400000,1.0,-50.000000,-139.000000,0.000000,8.400000,27.000000,4.800000,0.000000,0.000000,0.000000
25%,15.000000,2.000000,5.000000,1080.000000,1080.000000,16.453392,120.591317,1360.000000,5.000000,3.000000,1.0,9.000000,-6.250000,141.000000,18.000000,45.000000,6.400000,0.000000,0.000000,0.000000
50%,31.000000,3.000000,5.000000,1080.000000,1080.000000,16.453633,120.591589,1363.000000,9.000000,3.800000,1.0,22.000000,0.000000,210.000000,18.600000,57.000000,6.800000,1.000000,1.000000,0.000000
75%,68.000000,5.000000,10.000000,1920.000000,1080.000000,16.623848,120.758340,2421.000000,15.000000,6.500000,1.0,33.000000,4.000000,290.250000,20.300000,67.000000,6.800000,1.000000,2.000000,0.000000
max,185.000000,11.000000,11.000000,1920.000000,1080.000000,16.631366,120.763190,2442.000000,65.000000,344.000000,1.0,76.000000,52.000000,359.000000,23.800000,79.000000,7.200000,2.000000,2.000000,1.000000



DataFrame Information for df_cleaned:
<class 'pandas.core.frame.DataFrame'>
Index: 1636 entries, 0 to 1775
Data columns (total 31 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   uuid                 1636 non-null   object 
 1   spot_number          1636 non-null   int64  
 2   shot_number          1636 non-null   int64  
 3   shots_in_spot        1636 non-null   int64  
 4   image_filename       1636 non-null   object 
 5   image_width          1428 non-null   float64
 6   image_height         1428 non-null   float64
 7   image_quality        1428 non-null   object 
 8   capture_datetime     1636 non-null   object 
 9   latitude             1189 non-null   float64
 10  longitude            1189 non-null   float64
 11  altitude_m           1189 non-null   float64
 12  altitude_accuracy_m  1189 non-null   float64
 13  gps_accuracy_m       1189 non-null   float64
 14  gps_reading_count    1189 non-null   float64
 15  came

In [55]:
print("Shape of df_cleaned:")
print(df_cleaned.shape)

print("\nFirst 5 rows of df_cleaned:")
display(df_cleaned.head())

print("\nColumns of df_cleaned:")
print(df_cleaned.columns.tolist())

Shape of df_cleaned:
(1636, 31)

First 5 rows of df_cleaned:


,uuid,spot_number,shot_number,shots_in_spot,image_filename,image_width,image_height,image_quality,capture_datetime,latitude,...,crops,temperature_c,humidity_percent,notes,device_id,capture_mode,ph,k,p,n
0,aeeed204,1,1,10,images/2026/01/21/latrinidad_balili_bsuback_20...,1080.0,1080.0,1080p,2026-01-21T00:31:27.435Z,16.453933,...,"cabbage,custom_chinese_cabbage",16.9,79.0,NaN,TECNO_TECNO_CM8_5EAA20,field,6.4,1.0,0.0,0.0
1,a18676dc,1,2,10,images/2026/01/21/latrinidad_balili_bsuback_20...,1080.0,1080.0,1080p,2026-01-21T00:31:30.356Z,16.453933,...,"cabbage,custom_chinese_cabbage",16.9,79.0,NaN,TECNO_TECNO_CM8_5EAA20,field,6.4,1.0,0.0,0.0
2,f781136b,1,3,10,images/2026/01/21/latrinidad_balili_bsuback_20...,1080.0,1080.0,1080p,2026-01-21T00:31:33.202Z,16.453933,...,"cabbage,custom_chinese_cabbage",16.9,79.0,NaN,TECNO_TECNO_CM8_5EAA20,field,6.4,1.0,0.0,0.0
3,7cfb041f,1,4,10,images/2026/01/21/latrinidad_balili_bsuback_20...,1080.0,1080.0,1080p,2026-01-21T00:31:37.907Z,16.453933,...,"cabbage,custom_chinese_cabbage",16.9,79.0,NaN,TECNO_TECNO_CM8_5EAA20,field,6.4,1.0,0.0,0.0
4,db27f1eb,1,5,10,images/2026/01/21/latrinidad_balili_bsuback_20...,1080.0,1080.0,1080p,2026-01-21T00:31:40.736Z,16.453933,...,"cabbage,custom_chinese_cabbage",16.9,79.0,NaN,TECNO_TECNO_CM8_5EAA20,field,6.4,1.0,0.0,0.0



Columns of df_cleaned:
['uuid', 'spot_number', 'shot_number', 'shots_in_spot', 'image_filename', 'image_width', 'image_height', 'image_quality', 'capture_datetime', 'latitude', 'longitude', 'altitude_m', 'altitude_accuracy_m', 'gps_accuracy_m', 'gps_reading_count', 'camera_pitch', 'camera_roll', 'camera_heading', 'municipality', 'barangay', 'farm_name', 'crops', 'temperature_c', 'humidity_percent', 'notes', 'device_id', 'capture_mode', 'ph', 'k', 'p', 'n']
